# Lab 05 · A council of architects: propose, challenge, agree

**Prepared by Mustafa Assaf** · Lecture 2: Chain, act, debate · 15–20 minute lab

## Three architects, one decision

**So far:** in Lab 03, one architect's chain chose microservices for the Payments platform. It noticed the operations cost, but no one challenged the choice.

**New:** an *architecture council*. Three architects with different stances, each running on a different **model family**, answer **alone and in parallel**. Then a *chair* runs the debate by fixed rules:

1. It challenges every claim that cites no source, and gives the author one chance to revise.
2. It counts the votes of grounded proposals; a two-thirds majority decides.
3. It records the dissent, because a minority view can be right.
4. With no majority, it escalates to the human instead of guessing.

Every message is a small dictionary with a `round`, a `speaker`, and a `kind`. The list of all messages, the *transcript*, is the council's shared state: it shows who said what, based on which source.

The chair here is plain code, so its rules are easy to read and test. A chair can also be an agent, but keep its rules (who speaks, how many rounds, what counts as a majority) in code.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/dbe84948102e503e581fd5236bb7e67e63dba56c.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · Shared sources and a grounding rule

All architects see the same sources: the course policy R1 and three facts from the brief. Equal evidence keeps the debate fair. In your project, retrieve the sources with Lab 02's vector database.

`grounded` is the chair's evidence rule: a claim counts only if it cites at least one source, and every cited source exists.

In [ ]:
sources = {  # every architect sees the same evidence, so the debate is fair
    "R1": "Choose microservices only when the team has 8+ people and services must deploy independently.",
    "F1": "The Payments platform has 12 developers in 3 squads.",
    "F2": "Each squad wants to release on its own schedule.",
    "F3": "Only one engineer runs operations.",
}
evidence = "\n".join(f"[{sid}] {text}" for sid, text in sources.items())

def grounded(message):
    """A claim counts only if it cites at least one source, and every cited source exists."""
    return bool(message["evidence"]) and set(message["evidence"]) <= set(sources)

print(grounded({"evidence": ["R1", "F2"]}), grounded({"evidence": []}), grounded({"evidence": ["R9"]}))

## Step 2 · Three architects on three model families

The architects share one job and one set of sources, but each has its own stance and its own model family. Three personas on one model are not enough: one model brings one set of blind spots, so the course project requires at least three families. `asks` holds one `ask` function per architect, each with its own call budget. `say` builds every prompt the same way: the role and stance, the sources, and then the task.

Practice mode needs no keys. For live mode, FreeInference serves the Qwen family. Point the other two architects at a second OpenAI-compatible service, such as your university gateway: replace the placeholder URL, model ids, and secret name, and save that key in Colab Secrets. The second assert refuses to run live while any placeholder (anything containing `example`) remains. The first assert checks only the family labels, so make sure the models behind them really differ.

In [ ]:
council = {  # same job and sources; a different stance and model family for each architect
    "Pragmatist": {"family": "Qwen", "stance": "Prefer the simplest design that meets the requirements."},
    "Guardian": {"family": "Llama", "stance": "Prefer low-risk designs; weigh operations and failures."},
    "Innovator": {"family": "Mistral", "stance": "Prefer designs that let each squad change and release alone."},
}
GATEWAY = dict(base_url="https://gateway.example/v1", secret_name="GATEWAY_API_KEY")  # your second service
live = {"Pragmatist": dict(model="qwen3.6-35b"),  # FreeInference serves the Qwen family
        "Guardian": dict(GATEWAY, model="example-llama-model"), "Innovator": dict(GATEWAY, model="example-mistral")}
assert len({a["family"] for a in council.values()}) >= 3, "The council needs at least three model families."
assert MODE == "practice" or "example" not in str(live), "Replace the placeholder live settings first."
asks = {name: make_ask(mode=MODE, **live[name]) for name in council}

def say(name, task, example):
    """Ask one architect: its role and stance, the shared sources, then the task."""
    role = f"You are the {name} on an architecture council. {council[name]['stance']}"
    return asks[name](f"{role}\nSources:\n{evidence}\n{task}", example=example)

## Step 3 · Propose alone, in parallel

Each architect answers without seeing the others. Models tend to agree with what they have just read, so a proposal written after reading another one is not an independent opinion. `ThreadPoolExecutor` starts the three calls at the same time. (The course client limits how many requests go to one service at once. FreeInference's free tier takes one at a time, so architects that share it wait their turn; architects on different services really do run in parallel.) The chair turns each JSON reply into a message by adding the round, the speaker, and the kind.

In practice mode, the Innovator's scripted claim cites no source. Step 4 shows what the chair does with it.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor

REPLY = 'Reply with JSON only, no code fences: {"option": "...", "claim": "one sentence", "evidence": ["ids"]}'
scripted = {
    "Pragmatist": '{"option": "microservices", "claim": "Squads that release alone meet R1.", "evidence": ["R1", "F2"]}',
    "Guardian": '{"option": "modular monolith", "claim": "One engineer cannot run many services.", "evidence": ["F3"]}',
    "Innovator": '{"option": "microservices", "claim": "Microservices make payments faster.", "evidence": []}',
}

def propose(name):
    """One architect answers alone: the prompt never contains another proposal."""
    task = f"Choose modular monolith or microservices for the Payments platform. {REPLY}"
    return {"round": 1, "speaker": name, "kind": "proposal", **json.loads(say(name, task, scripted[name]))}

with ThreadPoolExecutor() as pool:  # the three architects work at the same time
    proposals = {message["speaker"]: message for message in pool.map(propose, council)}
for m in proposals.values():
    print(f"{m['speaker']:<11} {m['option']:<17} {str(m['evidence']):<19} {m['claim']}")

## Step 4 · Challenge unsupported claims

The chair checks every proposal with `grounded`. For a claim without a source, it picks a critic, preferring an architect who chose a different option, and asks for a one-sentence challenge. The author gets one chance to revise: it can change its option, or keep it with a claim that cites sources. Allowing one challenge per claim keeps the debate bounded, so it always ends. Each message is appended to `transcript`.

In [ ]:
transcript = list(proposals.values())  # shared state: every message, in order
critique = "Which source says that? Calls between services add network hops to every payment."
revision = '{"option": "microservices", "claim": "Squads that release alone meet R1.", "evidence": ["R1", "F2"]}'
for name, m in list(proposals.items()):
    if grounded(m):
        continue
    others = [n for n in council if n != name]
    critic = next((n for n in others if proposals[n]["option"] != m["option"]), others[0])  # prefer an opponent
    challenge = say(critic, f"In one sentence, challenge this claim by the {name}: {m['claim']}", critique)
    transcript.append({"round": 2, "speaker": critic, "kind": "challenge", "target": name, "text": challenge})
    task = f"You claimed: {m['claim']}\nChallenge: {challenge}\nRevise your proposal. {REPLY}"
    proposals[name] = {"round": 2, "speaker": name, "kind": "revision", **json.loads(say(name, task, revision))}
    transcript.append(proposals[name])
    print(f"{critic} challenges {name}: {challenge}")
    print(f"{name} revises: {proposals[name]['option']}, {proposals[name]['claim']} {proposals[name]['evidence']}")

## Step 5 · Count grounded votes, keep the dissent

Only grounded proposals vote. With a two-thirds majority, the chair decides and keeps every dissenting proposal, with its sources, in the outcome. Without one, the chair escalates to the human. Printing the transcript replays the whole debate.

In [ ]:
from collections import Counter

votes = Counter(m["option"] for m in proposals.values() if grounded(m))  # ungrounded claims do not vote
option, count = votes.most_common(1)[0] if votes else (None, 0)
decided = 3 * count >= 2 * len(council)  # the stopping rule: a two-thirds majority
outcome = {"round": 3, "speaker": "Chair", "kind": "decision" if decided else "escalation",  # no guessing
           "option": option if decided else None, "votes": dict(votes),
           "dissent": [m for m in proposals.values() if decided and m["option"] != option]}
transcript.append(outcome)
for m in transcript:
    print(f"round {m['round']}  {m['speaker']:<11} {m['kind']:<10} {m.get('text') or m['option']}")
print("Votes:", outcome["votes"], "| dissent:", [(m["speaker"], m["claim"]) for m in outcome["dissent"]])

## What you should see

Step 1 prints `True False False`. Step 3 prints three proposals: two for microservices and one for the modular monolith, and the Innovator's evidence list is empty. In Step 4, the Guardian, who chose the other option, challenges the Innovator, which then keeps microservices with a claim that cites R1 and F2. Step 5 replays six messages over three rounds and ends with a decision for microservices, 2 votes to 1. The Guardian's dissent, which cites F3, is kept.

The council used five model calls: three proposals, one challenge, and one revision.

## Try it

1. **Deadlock.** Make the scripted revision cite no source (`"evidence": []`). The Innovator's vote no longer counts, the grounded votes are tied 1 to 1, and the chair escalates.
2. **A made-up source.** Make the Guardian cite `["F9"]`. Who challenges the Guardian now, and why does the scripted revision change the outcome?
3. **Peeking.** Discuss: what would change if `propose` added the Pragmatist's proposal to the other prompts?
4. **Council against one model.** In live mode with three families, compare each architect's first proposal with the council's decision. Did the debate change anyone's mind? Did it change the decision?
5. **One family, three personas.** In live mode, give all three architects the same model. Do the stances alone produce any disagreement?

## Takeaway

Independent proposals keep different views alive until the debate starts. Challenges turn disagreement into evidence: a claim without a source must be revised or it does not vote. A chair with fixed rules makes sure the debate ends, with a decision and its dissent, or with an escalation to the human. Lab 06 gives this decision to the human architect. For self-study, Lab 11 runs an agent behind the A2A protocol and Lab 12 builds a check-and-retry loop with LangGraph: two ways to connect and organize your council.

## Read further

- [Building effective agents: parallelization and orchestration](https://www.anthropic.com/research/building-effective-agents)
- [Du et al., Improving factuality and reasoning through multiagent debate (2023)](https://arxiv.org/abs/2305.14325)
- [Sharma et al., Towards understanding sycophancy in language models (2023)](https://arxiv.org/abs/2310.13548)